# ViBioMIR snapshot: restore and query recheck

Attach only the dense snapshot notebook output version 1. This CPU notebook verifies its SHA-256, restores 4,936,835 points and checks the existing HNSW, FP16 vectors, source payloads and queries. No embedding, upsert or graph rebuild.

The original query exceeded Qdrant's default 60-second search deadline. We test that default once and then explicitly set the server request timeout to 300 seconds. This is a functional stored-vector test, not a semantic retrieval benchmark.


In [ ]:
import importlib.util
import json
from pathlib import Path
import subprocess
import sys

packages = {'numpy': 'numpy', 'pyarrow': 'pyarrow', 'requests': 'requests', 'psutil': 'psutil'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
print('CPU-only snapshot check:', sys.version)


In [ ]:
CONFIG = {'input_root': '/kaggle/input', 'expected_chunks': 4936835, 'source_sha256': 'a45123b3de7a9b048f83418515a3ec6b3dd09e1e4491dd361429b787659a3e7f', 'profile_id': '5d92039674725126e3757ac85a2fe632ed301cc19eeab4db0ddc2e6df6519087', 'dimension': 1024, 'batch_size': 128, 'collection': 'vibiomir_qwen3_06b_1024_fp16_dense_recheck', 'state_dir': '/tmp/vibiomir_qdrant_recheck_state', 'binary': '/tmp/vibiomir_qdrant_bin/qdrant', 'port': 6333, 'qdrant_url': 'http://127.0.0.1:6333', 'hnsw_m': 32, 'ef_construction': 100, 'indexing_threshold_kb': 10000, 'index_timeout_seconds': 18000, 'index_poll_seconds': 15, 'ingest_stop_seconds': 14400, 'output': '/kaggle/working/vibiomir_snapshot_recheck', 'output_budget_root': '/kaggle/working', 'output_limit_bytes': 19327352832, 'resume_from': None, 'input_notebooks': [{'ref': 'nhminh107/vibiomir-qwen3-06b-part1-t4x2-20261024', 'version': 1}, {'ref': 'nhminh107/vibiomir-qwen3-06b-part2-t4x2-20261024', 'version': 1}, {'ref': 'nhminh107/vibiomir-qwen3-06b-part3-t4x2-20261024', 'version': 3}], 'source_kernel': 'nhminh107/vibiomir-qwen3-dense-fp16-hnsw-20261025/1', 'search_timeout_seconds': 300}
MODULES = {'read_vibiomir_outputs.py': '"""Read embedding outputs through their manifests, with bounded memory."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport uuid\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef discover_outputs(input_root="/kaggle/input", required_parts=(1, 2, 3)):\n    manifests = {}\n    required_parts = sorted(required_parts)\n    for path in Path(input_root).rglob("manifest.json"):\n        manifest = json.loads(path.read_text())\n        if manifest.get("schema_version") != "vibiomir-embedding-v1":\n            continue\n        part = manifest["part"]\n        if part not in required_parts:\n            continue\n        if part in manifests:\n            raise ValueError(f"Part {part} attached more than once; keep one completed version per part")\n        if not manifest["complete"] or manifest["rows"] != manifest["expected_rows"]:\n            raise ValueError(f"Part {part} is incomplete: {path}")\n        manifests[part] = (path, manifest)\n    if sorted(manifests) != required_parts:\n        raise ValueError(f"Add Input completed parts {required_parts}; found {sorted(manifests)}")\n    first = manifests[required_parts[0]][1]\n    profile_id = hashlib.sha256(json.dumps(first["encoder"], sort_keys=True).encode()).hexdigest()\n    files = []\n    for part, (manifest_path, manifest) in sorted(manifests.items()):\n        if (manifest["encoder"] != first["encoder"] or manifest["source"] != first["source"]\n                or manifest["profile_id"] != profile_id):\n            raise ValueError("Parts use different encoder profiles or source data")\n        ranges = [item for item in manifest["source"]["shards"] if item["part"] == part]\n        expected_rows = sum(item["rows"] for item in ranges)\n        if manifest["rows"] != expected_rows:\n            raise ValueError(f"Part {part} row count differs from source partition")\n        next_line = min(item["start_line"] for item in ranges)\n        end_line = max(item["end_line"] for item in ranges)\n        for receipt in sorted(manifest["files"], key=lambda item: item["start_line"]):\n            if (Path(receipt["file"]).name != receipt["file"] or receipt["part"] != part\n                    or receipt["start_line"] != next_line\n                    or receipt["end_line"] - receipt["start_line"] + 1 != receipt["rows"]\n                    or receipt["profile_id"] != profile_id\n                    or receipt["source_sha256"] != manifest["source"]["source_sha256"]):\n                raise ValueError(f"Invalid file/source mapping in part {part}")\n            path = manifest_path.parent / receipt["file"]\n            if not path.is_file():\n                raise FileNotFoundError(f"Manifest references a missing file: {path}")\n            parquet = pq.ParquetFile(path)\n            metadata = json.loads(parquet.schema_arrow.metadata[b"vibiomir_embedding"])\n            vector_type = parquet.schema_arrow.field("vector").type\n            if (parquet.metadata.num_rows != receipt["rows"]\n                    or not pa.types.is_fixed_size_list(vector_type)\n                    or vector_type.list_size != manifest["encoder"]["dimension"]\n                    or vector_type.value_type != pa.float32()\n                    or metadata["profile_id"] != profile_id\n                    or metadata["source_sha256"] != receipt["source_sha256"]):\n                raise ValueError(f"Parquet schema/metadata differs: {path}")\n            files.append(dict(path=str(path), receipt=receipt))\n            next_line = receipt["end_line"] + 1\n        if next_line != end_line + 1:\n            raise ValueError(f"Missing source rows in part {part}")\n    return dict(parts=required_parts, rows=sum(item[1]["rows"] for item in manifests.values()),\n                source_rows=first["source"]["total_rows"], source_sha256=first["source"]["source_sha256"],\n                encoder=first["encoder"], profile_id=profile_id,\n                full_corpus=len(required_parts) == first["source"]["num_parts"], files=files)\n\n\ndef iter_embedding_batches(catalog, batch_size=128, columns=None):\n    for item in catalog["files"]:\n        yield from pq.ParquetFile(item["path"]).iter_batches(batch_size=batch_size, columns=columns)\n\n\ndef verify_outputs(catalog, verify_checksums=True, metadata_sample_per_file=128):\n    scanned_rows = sampled_rows = 0\n    namespace = uuid.uuid5(uuid.NAMESPACE_URL, "https://vibiomir.org/chunks")\n    for file_index, item in enumerate(catalog["files"], 1):\n        path, receipt = Path(item["path"]), item["receipt"]\n        if verify_checksums and file_hash(path) != receipt["sha256"]:\n            raise ValueError(f"Checksum differs: {path}")\n        parquet = pq.ParquetFile(path)\n        next_line = receipt["start_line"]\n        for batch in parquet.iter_batches(batch_size=2048, columns=["source_line", "vector"]):\n            lines = batch.column(0).to_numpy()\n            vectors = batch.column(1).flatten().to_numpy().reshape(len(lines), catalog["encoder"]["dimension"])\n            if (not np.array_equal(lines, np.arange(next_line, next_line + len(lines)))\n                    or not np.isfinite(vectors).all()\n                    or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=2e-5)):\n                raise ValueError(f"Invalid vector values or source order: {path}")\n            next_line += len(lines)\n            scanned_rows += len(lines)\n        if next_line != receipt["end_line"] + 1:\n            raise ValueError(f"Source coverage differs: {path}")\n        sample = next(parquet.iter_batches(batch_size=metadata_sample_per_file)).to_pylist()\n        for row in sample:\n            payload = json.loads(row["payload_json"])\n            provenance = payload["_embedding"]\n            content_hash = hashlib.sha256(payload["content"].encode()).hexdigest()\n            if (payload["chunk_id"] != row["chunk_id"] or payload["doc_id"] != row["doc_id"]\n                    or payload["content"] != row["content"]\n                    or provenance["source_line"] != row["source_line"]\n                    or provenance["source_byte"] != row["source_byte"]\n                    or content_hash != row["content_sha256"]\n                    or provenance["content_sha256"] != content_hash\n                    or provenance["profile_id"] != catalog["profile_id"]\n                    or str(uuid.uuid5(namespace, row["chunk_id"])) != row["point_id"]):\n                raise ValueError(f"Vector/payload ID mapping differs: {path}")\n            sampled_rows += 1\n        if file_index % 25 == 0 or file_index == len(catalog["files"]):\n            print(json.dumps(dict(files_checked=file_index, files_total=len(catalog["files"]),\n                                  vector_rows_checked=scanned_rows)), flush=True)\n    if scanned_rows != catalog["rows"]:\n        raise ValueError("Total scanned rows differs from manifests")\n    return dict(passed=True, parts=catalog["parts"], full_corpus=catalog["full_corpus"],\n                parquet_files=len(catalog["files"]), vector_rows_checked=scanned_rows,\n                metadata_rows_sampled=sampled_rows, all_file_checksums_verified=verify_checksums,\n                dimension=catalog["encoder"]["dimension"], dtype="float32", normalized=True,\n                source_sha256=catalog["source_sha256"], profile_id=catalog["profile_id"])\n', 'restore_vibiomir_qdrant.py': '"""Verify, unpack and query the standalone ViBioMIR dense snapshot."""\n\nimport argparse\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shlex\nimport shutil\nfrom urllib.parse import quote\n\nimport numpy as np\nimport requests\n\n\ndef checksum(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nclass QdrantDenseClient:\n    def __init__(self, url, collection, vector_name="qwen3_dense", api_key=None,\n                 search_timeout=None, request_timeout=300):\n        self.url = url.rstrip("/")\n        self.collection = collection\n        self.vector_name = vector_name\n        self.search_timeout = search_timeout\n        self.request_timeout = request_timeout\n        self.session = requests.Session()\n        if api_key:\n            self.session.headers["api-key"] = api_key\n\n    @classmethod\n    def from_manifest(cls, path, url="http://127.0.0.1:6333", api_key=None):\n        manifest = json.loads(Path(path).read_text())\n        if not manifest["complete"]:\n            raise ValueError("Recovery snapshot: resume the notebook before production search")\n        return cls(url, manifest["collection"], manifest["vector_schema"]["dense"], api_key)\n\n    def request(self, method, suffix="", body=None):\n        name = quote(self.collection, safe="")\n        if suffix == "/points/query" and self.search_timeout is not None:\n            suffix += f"?timeout={self.search_timeout}"\n        response = self.session.request(method, f"{self.url}/collections/{name}{suffix}",\n                                        json=body, timeout=self.request_timeout)\n        response.raise_for_status()\n        return response.json()["result"]\n\n    def search_dense(self, query_vector, top_k=10, filters=None, hnsw_ef=128):\n        body = {"query": list(query_vector), "using": self.vector_name, "limit": top_k,\n                "with_payload": True, "params": {"hnsw_ef": hnsw_ef}}\n        if filters is not None:\n            body["filter"] = filters\n        return self.request("POST", "/points/query", body)["points"]\n\n    def get_points(self, ids, with_vectors=False):\n        return self.request("POST", "/points", {"ids": ids, "with_payload": True,\n                                                "with_vector": with_vectors})\n\n    def lookup_chunk(self, chunk_id):\n        result = self.request("POST", "/points/scroll", {\n            "limit": 1, "with_payload": True,\n            "filter": {"must": [{"key": "chunk_id", "match": {"value": chunk_id}}]},\n        })\n        return result["points"]\n\n    def scroll_document(self, doc_id, limit=100, offset=None):\n        body = {"limit": limit, "with_payload": True,\n                "filter": {"must": [{"key": "doc_id", "match": {"value": doc_id}}]}}\n        if offset is not None:\n            body["offset"] = offset\n        return self.request("POST", "/points/scroll", body)\n\n    def close(self):\n        self.session.close()\n\n\ndef verify_collection(url, manifest, samples, collection=None, api_key=None, search_timeout=None):\n    client = QdrantDenseClient(url, collection or manifest["collection"],\n                               manifest["vector_schema"]["dense"], api_key,\n                               search_timeout=search_timeout,\n                               request_timeout=max(300, (search_timeout or 0) + 30))\n    try:\n        count = client.request("POST", "/points/count", {"exact": True})["count"]\n        if count != manifest["points"]:\n            raise ValueError("Exact point count differs from the snapshot manifest")\n        info = client.request("GET")\n        params = info["config"]["params"]\n        vectors = params["vectors"]\n        expected = {"size": manifest["signature"]["dimension"], "distance": "Cosine",\n                    "datatype": "float16", "on_disk": True}\n        if (set(vectors) != {client.vector_name} or params.get("sparse_vectors")\n                or any(vectors[client.vector_name].get(k) != v for k, v in expected.items())\n                or not params.get("on_disk_payload")):\n            raise ValueError("Collection is not the expected disk-backed FP16 dense schema")\n        for field, kind in manifest["payload_indexes"].items():\n            if info["payload_schema"].get(field, {}).get("data_type") != kind:\n                raise ValueError(f"Missing or different payload index: {field}")\n        hnsw = info["config"]["hnsw_config"]\n        if manifest["complete"] and (hnsw["m"] != manifest["signature"]["hnsw_m"]\n                or hnsw["ef_construct"] != manifest["signature"]["ef_construction"]):\n            raise ValueError("Restored HNSW configuration differs")\n        if not samples:\n            raise ValueError("No source samples available to verify payload/vector preservation")\n        points = {str(p["id"]): p for p in client.get_points([s["id"] for s in samples], True)}\n        for sample in samples:\n            point = points.get(sample["id"])\n            if point is None or point["payload"] != sample["payload"]:\n                raise ValueError("Restored source payload does not round-trip exactly")\n            original = np.asarray(sample["vector"], dtype=np.float32)\n            stored = np.asarray(point["vector"][client.vector_name], dtype=np.float32)\n            if (original.shape != stored.shape or not np.isfinite(stored).all()\n                    or not np.allclose(original, stored, rtol=0.003, atol=0.0005)):\n                raise ValueError("Restored FP16 vector differs beyond the allowed rounding tolerance")\n        for sample in samples[:3]:\n            payload = sample["payload"]\n            hits = client.search_dense(sample["vector"], top_k=3)\n            if not hits or hits[0]["score"] < 0.99:\n                raise ValueError("Dense stored-vector similarity check failed")\n            found = client.lookup_chunk(payload["chunk_id"])\n            if not found or found[0]["payload"] != payload:\n                raise ValueError("chunk_id lookup lost the original payload")\n            docs = client.scroll_document(payload["doc_id"], limit=3)["points"]\n            if not docs or any(p["payload"]["doc_id"] != payload["doc_id"] for p in docs):\n                raise ValueError("doc_id lookup failed")\n        return {"verified": True, "points": count, "payload_samples": len(samples),\n                "dense": True, "fp16": True, "payload_lookup": True,\n                "status": info["status"], "indexed_vectors_count": info.get("indexed_vectors_count", 0)}\n    finally:\n        client.close()\n\n\ndef unpack(folder):\n    folder = Path(folder)\n    manifest = json.loads((folder / "snapshot_manifest.json").read_text())\n    snapshot = folder / manifest["snapshot_file"]\n    if checksum(snapshot) != manifest["snapshot_sha256"]:\n        raise ValueError("Snapshot checksum mismatch")\n    raw = snapshot.with_suffix("") if snapshot.suffix == ".gz" else snapshot\n    if snapshot.suffix == ".gz" and not raw.exists():\n        if shutil.disk_usage(folder).free < manifest["raw_snapshot_bytes"] + 1024**3:\n            raise RuntimeError("Not enough disk space to unpack the snapshot")\n        partial = raw.with_suffix(raw.suffix + ".partial")\n        with gzip.open(snapshot, "rb") as source, partial.open("wb") as destination:\n            shutil.copyfileobj(source, destination, 8 * 1024**2)\n        if checksum(partial) != manifest["raw_snapshot_sha256"]:\n            raise ValueError("Unpacked snapshot checksum mismatch")\n        partial.replace(raw)\n    if checksum(raw) != manifest["raw_snapshot_sha256"]:\n        raise ValueError("Raw snapshot checksum mismatch")\n    return manifest, raw\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--folder", type=Path, default=Path.cwd())\n    parser.add_argument("--verify-url")\n    args = parser.parse_args()\n    folder = args.folder.resolve()\n    if args.verify_url:\n        manifest = json.loads((folder / "snapshot_manifest.json").read_text())\n        if not manifest["complete"]:\n            raise ValueError("Recovery snapshot is incomplete; resume the notebook first")\n        samples = json.loads((folder / "verification_samples.json").read_text())\n        print(json.dumps(verify_collection(args.verify_url, manifest, samples), indent=2))\n        return\n    manifest, raw = unpack(folder)\n    name = quote(manifest["collection"], safe="")\n    version = manifest["qdrant_version"]\n    if not manifest["complete"]:\n        print("RECOVERY ONLY: attach this export to the notebook and set resume_from.")\n    print("Use empty qdrant_storage; keep the snapshot and storage on disk during recovery.")\n    print("docker run -d --name vibiomir-qdrant -p 127.0.0.1:6333:6333 "\n          "-e QDRANT__STORAGE__LOW_MEMORY_MODE=no_populate "\n          "-e QDRANT__STORAGE__SNAPSHOTS_PATH=/qdrant/storage/snapshots "\n          f"-v {shlex.quote(str(folder / \'qdrant_storage\') + \':/qdrant/storage\')} "\n          f"-v {shlex.quote(str(folder) + \':/qdrant/import:ro\')} qdrant/qdrant:v{version}")\n    print("# Wait until curl --fail http://127.0.0.1:6333/readyz succeeds, then:")\n    print("docker exec vibiomir-qdrant mkdir -p /qdrant/storage/snapshots")\n    print(f"docker exec vibiomir-qdrant cp --reflink=auto {shlex.quote(\'/qdrant/import/\' + raw.name)} /qdrant/storage/snapshots/")\n    body = json.dumps({"location": f"file:///qdrant/storage/snapshots/{raw.name}", "priority": "snapshot"})\n    print(f"curl --fail -X PUT \'http://127.0.0.1:6333/collections/{name}/snapshots/recover?wait=true\' "\n          f"-H \'Content-Type: application/json\' --data {shlex.quote(body)}")\n    print(f"python restore_qdrant_snapshot.py --folder {shlex.quote(str(folder))} --verify-url http://127.0.0.1:6333")\n\n\nif __name__ == "__main__":\n    main()\n', 'vibiomir_qdrant_runtime.py': '"""Stream the three ViBioMIR embedding outputs into a portable FP16 dense collection."""\n\nimport argparse\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport socket\nimport subprocess\nimport tarfile\nimport time\nimport uuid\n\nimport numpy as np\nimport psutil\nimport pyarrow.parquet as pq\nimport requests\n\nif __package__:\n    from .read_vibiomir_outputs import discover_outputs\n    from .restore_vibiomir_qdrant import verify_collection\nelse:\n    from read_vibiomir_outputs import discover_outputs\n    from restore_vibiomir_qdrant import verify_collection\n\nQDRANT_VERSION = "1.19.2"\nQDRANT_URL = f"https://github.com/qdrant/qdrant/releases/download/v{QDRANT_VERSION}/qdrant-x86_64-unknown-linux-musl.tar.gz"\nQDRANT_SHA256 = "50b253243309ed0ae50a19f678f0d0adcc4f0b567b3bd2c5c0c7e02fa8404bb6"\nVECTOR_NAME = "qwen3_dense"\nPAYLOAD_INDEXES = {"chunk_id": "keyword", "doc_id": "integer", "content_origin": "keyword"}\nNAMESPACE = uuid.UUID("ba368b92-593b-53e5-9fca-f889a83f6f8f")\n\ndef write_json(path: Path, value: dict) -> None:\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef checksum(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef output_budget(config: dict) -> int:\n    root = Path(config["output_budget_root"])\n    used = sum(path.stat().st_size for path in root.rglob("*") if path.is_file())\n    return max(0, min(config["output_limit_bytes"] - used,\n                      shutil.disk_usage(root).free - 512 * 1024**2))\n\n\ndef download_qdrant(config: dict) -> Path:\n    binary = Path(config["binary"])\n    if binary.exists():\n        return binary\n    binary.parent.mkdir(parents=True, exist_ok=True)\n    archive = binary.with_suffix(".tar.gz")\n    with requests.get(QDRANT_URL, stream=True, timeout=(30, 120)) as response:\n        response.raise_for_status()\n        with archive.open("wb") as stream:\n            for block in response.iter_content(1024**2):\n                stream.write(block)\n    if checksum(archive) != QDRANT_SHA256:\n        raise ValueError("Official Qdrant release archive SHA-256 mismatch")\n    with tarfile.open(archive) as bundle:\n        member = next(item for item in bundle.getmembers()\n                      if Path(item.name).name == "qdrant" and item.isfile())\n        with bundle.extractfile(member) as source, binary.open("wb") as destination:\n            shutil.copyfileobj(source, destination)\n    binary.chmod(0o755)\n    return binary\n\n\ndef api(config: dict, method: str, path: str, body=None, session=None, timeout=300):\n    connection = session or requests\n    response = connection.request(method, config["qdrant_url"] + path, json=body, timeout=timeout)\n    if not response.ok:\n        raise RuntimeError(f"Qdrant {method} {path}: HTTP {response.status_code}: {response.text[:1500]}")\n    return response.json().get("result")\n\n\ndef start_server(config: dict):\n    binary = download_qdrant(config)\n    port = config["port"]\n    with socket.socket() as probe:\n        if probe.connect_ex(("127.0.0.1", port)) == 0:\n            raise RuntimeError(f"Port {port} is already in use; refusing to attach to another database")\n    state = Path(config["state_dir"])\n    for directory in ("storage", "snapshots", "snapshot_temp"):\n        (state / directory).mkdir(parents=True, exist_ok=True)\n    server_config = {\n        "log_level": "WARN", "telemetry_disabled": True,\n        "storage": {"storage_path": str(state / "storage"),\n                    "snapshots_path": str(state / "snapshots"),\n                    "temp_path": str(state / "snapshot_temp"),\n                    "on_disk_payload": True, "low_memory_mode": "no_populate",\n                    "performance": {"max_search_threads": 2, "optimizer_cpu_budget": 2}},\n        "service": {"host": "127.0.0.1", "http_port": port, "grpc_port": port + 1,\n                    "max_request_size_mb": 64, "max_workers": 2},\n    }\n    # JSON syntax is valid YAML; Qdrant\'s loader expects a .yaml config path.\n    config_path = state / "qdrant_config.yaml"\n    write_json(config_path, server_config)\n    log = (Path(config["output"]) / "qdrant.log").open("a", encoding="utf-8")\n    process = subprocess.Popen([str(binary), "--config-path", str(config_path), "--disable-telemetry"],\n                               cwd=state, stdout=log, stderr=subprocess.STDOUT)\n    try:\n        for _ in range(120):\n            if process.poll() is not None:\n                raise RuntimeError("Qdrant exited during startup; inspect qdrant.log")\n            try:\n                response = requests.get(config["qdrant_url"] + "/readyz", timeout=1)\n                if response.ok:\n                    version = requests.get(config["qdrant_url"], timeout=5).json()["version"]\n                    if version != QDRANT_VERSION:\n                        raise RuntimeError(f"Expected Qdrant {QDRANT_VERSION}; server reports {version}")\n                    print(f"Qdrant {version} READY at {config[\'qdrant_url\']}", flush=True)\n                    return process, log\n            except requests.ConnectionError:\n                pass\n            time.sleep(0.5)\n        raise TimeoutError("Qdrant did not become ready in 60 seconds")\n    except BaseException:\n        stop_server(process, log)\n        raise\n\n\ndef stop_server(process, log) -> None:\n    if process.poll() is None:\n        process.terminate()\n        try:\n            process.wait(timeout=30)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n    log.close()\n\n\ndef count_points(config: dict) -> int:\n    return api(config, "POST", f"/collections/{config[\'collection\']}/points/count", {"exact": True})["count"]\n\n\ndef signature(config, catalog):\n    identity = {"source_sha256": catalog["source_sha256"], "profile_id": catalog["profile_id"],\n                "encoder": catalog["encoder"], "expected_chunks": catalog["rows"],\n                "dimension": catalog["encoder"]["dimension"], "collection": config["collection"],\n                "dense_datatype": "float16", "vector_name": VECTOR_NAME,\n                "hnsw_m": config["hnsw_m"], "ef_construction": config["ef_construction"],\n                "payload_indexes": PAYLOAD_INDEXES}\n    receipts = [item["receipt"] for item in catalog["files"]]\n    identity["files_sha256"] = hashlib.sha256(json.dumps(receipts, sort_keys=True).encode()).hexdigest()\n    return identity\n\n\ndef load_catalog(config):\n    catalog = discover_outputs(config["input_root"], required_parts=(1, 2, 3))\n    if (not catalog["full_corpus"] or catalog["rows"] != config["expected_chunks"]\n            or catalog["encoder"]["dimension"] != config["dimension"]\n            or catalog["encoder"]["vector_name"] != VECTOR_NAME\n            or catalog["source_sha256"] != config["source_sha256"]\n            or catalog["profile_id"] != config["profile_id"]):\n        raise ValueError("Inputs differ from the required full ViBioMIR source/encoder profile")\n    payload_bytes = 0\n    for index, item in enumerate(catalog["files"], 1):\n        path = Path(item["path"])\n        if checksum(path) != item["receipt"]["sha256"]:\n            raise ValueError(f"Parquet checksum mismatch: {path}")\n        parquet = pq.ParquetFile(path)\n        column = parquet.schema_arrow.get_field_index("payload_json")\n        payload_bytes += sum(parquet.metadata.row_group(i).column(column).total_uncompressed_size\n                             for i in range(parquet.num_row_groups))\n        if index % 25 == 0 or index == len(catalog["files"]):\n            print(f"INPUT VERIFIED: {index}/{len(catalog[\'files\'])} files; {catalog[\'rows\']:,} total vectors", flush=True)\n    catalog["payload_bytes"] = payload_bytes\n    return catalog\n\n\ndef check_resources(config, catalog):\n    state = Path(config["state_dir"])\n    mounts = [m for m in psutil.disk_partitions(all=True) if state.resolve().is_relative_to(Path(m.mountpoint))]\n    kind = max(mounts, key=lambda m: len(m.mountpoint)).fstype if mounts else "unknown"\n    if kind in {"tmpfs", "ramfs"}:\n        raise RuntimeError(f"Qdrant state must use disk, not {kind}: {state}")\n    vectors = catalog["rows"] * config["dimension"] * 2\n    graph = catalog["rows"] * (8 * config["hnsw_m"] + 128)\n    collection = vectors + graph + catalog["payload_bytes"]\n    required = 4 * collection + 4 * 1024**3\n    free = shutil.disk_usage(state).free\n    budget = output_budget(config)\n    report = {"state_filesystem": kind, "state_free_bytes": free,\n              "estimated_collection_bytes": collection, "required_state_bytes": required,\n              "output_available_bytes": budget, "raw_fp16_vector_bytes": vectors,\n              "estimated_graph_bytes": graph, "parquet_payload_uncompressed_bytes": catalog["payload_bytes"],\n              "ram_available_bytes": psutil.virtual_memory().available}\n    write_json(Path(config["output"]) / "preflight.json", report)\n    print("RESOURCES:", json.dumps(report), flush=True)\n    if free < required:\n        raise RuntimeError("Insufficient disk for collection, snapshots, rewrites and restore verification")\n    if budget < vectors + 512 * 1024**2:\n        raise RuntimeError("Output budget cannot hold the FP16 vectors plus export headroom")\n    if min(psutil.virtual_memory().available, psutil.virtual_memory().total) < 3 * 1024**3:\n        raise RuntimeError("Less than 3 GiB RAM available for the disk-backed CPU pipeline")\n\n\ndef create_collection(config):\n    name = config["collection"]\n    existing = api(config, "GET", "/collections")["collections"]\n    if not any(c["name"] == name for c in existing):\n        api(config, "PUT", f"/collections/{name}", {\n            "vectors": {VECTOR_NAME: {"size": config["dimension"], "distance": "Cosine",\n                                       "on_disk": True, "datatype": "float16"}},\n            "on_disk_payload": True, "shard_number": 1,\n            "hnsw_config": {"m": 0, "on_disk": True, "max_indexing_threads": 2},\n            "optimizers_config": {"default_segment_number": 2, "max_segment_size": 500_000,\n                                  "indexing_threshold": config["indexing_threshold_kb"],\n                                  "max_optimization_threads": 1},\n        })\n    info = api(config, "GET", f"/collections/{name}")\n    params = info["config"]["params"]\n    expected = {"size": config["dimension"], "distance": "Cosine", "datatype": "float16", "on_disk": True}\n    if (set(params["vectors"]) != {VECTOR_NAME} or params.get("sparse_vectors")\n            or any(params["vectors"][VECTOR_NAME].get(k) != v for k, v in expected.items())\n            or not params.get("on_disk_payload")):\n        raise ValueError("Existing collection is not the expected disk-backed FP16 dense schema")\n    for field, kind in PAYLOAD_INDEXES.items():\n        api(config, "PUT", f"/collections/{name}/index?wait=true",\n            {"field_name": field, "field_schema": {"type": kind, "on_disk": True}})\n\n\ndef iter_points(path, receipt, dimension, batch_size, skip_rows=0):\n    seen = 0\n    for batch in pq.ParquetFile(path).iter_batches(batch_size=batch_size):\n        data = batch.to_pydict()\n        vectors = np.asarray(data["vector"], dtype=np.float32)\n        if (vectors.shape != (batch.num_rows, dimension) or not np.isfinite(vectors).all()\n                or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=2e-5, rtol=0)):\n            raise ValueError(f"Invalid embedding values in {path}")\n        points = []\n        for index in range(batch.num_rows):\n            row = {k: v[index] for k, v in data.items() if k != "vector"}\n            payload = json.loads(row["payload_json"])\n            provenance = payload["_embedding"]\n            source_line = receipt["start_line"] + seen + index\n            content_hash = hashlib.sha256(payload["content"].encode("utf-8")).hexdigest()\n            if (row["source_line"] != source_line or provenance["source_line"] != source_line\n                    or provenance["source_byte"] != row["source_byte"]\n                    or provenance["source_sha256"] != receipt["source_sha256"]\n                    or provenance["profile_id"] != receipt["profile_id"]\n                    or provenance["part"] != receipt["part"] or provenance["gpu"] != receipt["gpu"]\n                    or provenance["input_token_count"] != row["input_token_count"]\n                    or provenance["truncated"] is not False\n                    or any(payload[k] != row[k] for k in ("chunk_id", "doc_id", "content", "content_origin", "token_count"))\n                    or row["content_sha256"] != content_hash or provenance["content_sha256"] != content_hash\n                    or str(uuid.uuid5(NAMESPACE, payload["chunk_id"])) != row["point_id"]):\n                raise ValueError(f"Payload, point ID or provenance mismatch: {path}, line {source_line}")\n            if seen + index >= skip_rows:\n                points.append({"id": row["point_id"], "vector": {VECTOR_NAME: vectors[index].tolist()},\n                               "payload": payload})\n        seen += batch.num_rows\n        if points:\n            yield points, seen\n    if seen != receipt["rows"] or not 0 <= skip_rows <= seen:\n        raise ValueError("Parquet/checkpoint row count differs")\n\n\ndef upsert(config, points, session):\n    for attempt in range(4):\n        try:\n            response = session.put(f"{config[\'qdrant_url\']}/collections/{config[\'collection\']}/points?wait=true",\n                                   json={"points": points}, timeout=180)\n            if response.status_code in (408, 429, 500, 502, 503, 504):\n                if attempt == 3:\n                    response.raise_for_status()\n            else:\n                response.raise_for_status()\n                if response.json()["result"]["status"] != "completed":\n                    raise RuntimeError("Qdrant did not acknowledge completed upsert")\n                return\n        except (requests.ConnectionError, requests.Timeout):\n            if attempt == 3:\n                raise\n        time.sleep(2**attempt)\n\n\ndef checkpoint_path(config):\n    return Path(config["state_dir"]) / "ingestion_checkpoint.json"\n\n\ndef ingest(config, catalog, state, started):\n    initial_rows = state["rows"]\n    output = Path(config["output"])\n    samples_path = output / "verification_samples.json"\n    samples = json.loads(samples_path.read_text()) if samples_path.exists() else []\n    sample_stride = max(1, len(catalog["files"]) // 48)\n    with requests.Session() as session:\n        for file_index, item in enumerate(catalog["files"]):\n            if file_index < state["file_index"]:\n                continue\n            path, receipt = Path(item["path"]), item["receipt"]\n            skip = state["file_rows"] if file_index == state["file_index"] else 0\n            for points, end_row in iter_points(path, receipt, config["dimension"], config["batch_size"], skip):\n                if time.monotonic() - started > config["ingest_stop_seconds"]:\n                    raise TimeoutError("Controlled stop: preserve recovery snapshot before the session limit")\n                upsert(config, points, session)\n                # Store the sample before its checkpoint; replaying a batch remains idempotent.\n                if (file_index % sample_stride == 0 or not samples) and end_row - len(points) == 0:\n                    point = points[0]\n                    sample = {"id": point["id"], "payload": point["payload"], "vector": point["vector"][VECTOR_NAME]}\n                    if all(s["id"] != point["id"] for s in samples):\n                        samples.append(sample)\n                        write_json(samples_path, samples)\n                state.update(file_index=file_index, file_rows=end_row,\n                             rows=state["rows"] + len(points), phase="ingesting")\n                write_json(checkpoint_path(config), state)\n                if end_row == receipt["rows"] or state["rows"] % (config["batch_size"] * 100) == 0:\n                    seconds = max(0.001, time.monotonic() - started)\n                    print(json.dumps({"phase": "ingesting", "part": receipt["part"], "file": path.name,\n                                      "rows": state["rows"], "expected": catalog["rows"],\n                                      "new_rows_per_second": round((state["rows"] - initial_rows) / seconds, 1),\n                                      "elapsed_seconds": round(seconds, 1)}), flush=True)\n            state.update(file_index=file_index + 1, file_rows=0)\n            write_json(checkpoint_path(config), state)\n    if state["rows"] != catalog["rows"] or count_points(config) != catalog["rows"]:\n        raise ValueError("Committed rows or exact Qdrant count differs from the full corpus")\n    state["phase"] = "ingested"\n    write_json(checkpoint_path(config), state)\n\n\ndef wait_for_index(config):\n    name = config["collection"]\n    api(config, "PATCH", f"/collections/{name}", {\n        "hnsw_config": {"m": config["hnsw_m"], "ef_construct": config["ef_construction"],\n                        "on_disk": True, "max_indexing_threads": 2},\n    })\n    deadline = time.monotonic() + config["index_timeout_seconds"]\n    stable = 0\n    while time.monotonic() < deadline:\n        info = api(config, "GET", f"/collections/{name}")\n        if info["optimizer_status"] != "ok" or info["status"] == "red":\n            raise RuntimeError(f"Optimizer error: {info[\'optimizer_status\']}")\n        allowance = info["segments_count"] * math.ceil(config["indexing_threshold_kb"] * 1024 / (config["dimension"] * 2))\n        ready = info["status"] == "green" and info.get("indexed_vectors_count", 0) >= config["expected_chunks"] - allowance\n        stable = stable + 1 if ready else 0\n        print(json.dumps({"phase": "indexing", "status": info["status"], "points": info["points_count"],\n                          "indexed": info.get("indexed_vectors_count", 0), "segments": info["segments_count"]}), flush=True)\n        if stable >= 3:\n            return info\n        time.sleep(config["index_poll_seconds"])\n    raise TimeoutError("HNSW optimization timed out; export recovery, not a complete snapshot")\n\n\ndef snapshot_raw(config):\n    name = config["collection"]\n    before = count_points(config)\n    result = api(config, "POST", f"/collections/{name}/snapshots?wait=true", timeout=3600)\n    if count_points(config) != before:\n        raise ValueError("Collection changed while creating snapshot")\n    raw = Path(config["state_dir"]) / "snapshots" / name / Path(result["name"]).name\n    if not raw.is_file() or not tarfile.is_tarfile(raw):\n        raise ValueError("Qdrant snapshot is missing or invalid")\n    return raw, before\n\n\ndef snapshot_manifest(config, identity, state, points, complete):\n    return {"schema_version": "vibiomir-qdrant-dense-v1", "complete": complete,\n            "collection": config["collection"], "qdrant_version": QDRANT_VERSION,\n            "signature": identity, "points": points, "checkpoint": state.copy(),\n            "vector_schema": {"dense": VECTOR_NAME, "dense_datatype": "float16"},\n            "payload_indexes": PAYLOAD_INDEXES, "embedding": identity["encoder"],\n            "storage": {"dense_datatype": "float16", "on_disk": True},\n            "parent_mapping": {"document_key": "doc_id", "parent_id_available": False}}\n\n\ndef verify_snapshot_restore(config, raw, manifest, samples):\n    name = config["collection"] + "_restore_check"\n    if any(c["name"] == name for c in api(config, "GET", "/collections")["collections"]):\n        raise ValueError("Restore verification collection already exists")\n    api(config, "PUT", f"/collections/{name}/snapshots/recover?wait=true",\n        {"location": raw.resolve().as_uri(), "priority": "snapshot"}, timeout=3600)\n    try:\n        return verify_collection(config["qdrant_url"], manifest, samples, collection=name)\n    finally:\n        api(config, "DELETE", f"/collections/{name}?timeout=120")\n\n\ndef export_snapshot(config, identity, state, complete):\n    raw, points = snapshot_raw(config)\n    manifest = snapshot_manifest(config, identity, state, points, complete)\n    output = Path(config["output"])\n    samples = json.loads((output / "verification_samples.json").read_text())\n    manifest["restore_verification"] = verify_snapshot_restore(config, raw, manifest, samples)\n    manifest["raw_snapshot_sha256"] = checksum(raw)\n    manifest["raw_snapshot_bytes"] = raw.stat().st_size\n    prefix = "vibiomir_qwen3_dense" if complete else "RECOVERY_vibiomir_qwen3_dense"\n    source = raw\n    if raw.stat().st_size > output_budget(config):\n        source = Path(config["state_dir"]) / f"{prefix}.snapshot.gz"\n        print("Compressing snapshot losslessly outside Output", flush=True)\n        with raw.open("rb") as reader, gzip.open(source, "wb", compresslevel=3) as writer:\n            shutil.copyfileobj(reader, writer, 8 * 1024**2)\n        digest = hashlib.sha256()\n        with gzip.open(source, "rb") as reader:\n            for block in iter(lambda: reader.read(8 * 1024**2), b""):\n                digest.update(block)\n        if digest.hexdigest() != manifest["raw_snapshot_sha256"]:\n            raise ValueError("Compressed snapshot failed lossless verification")\n    if source.stat().st_size > output_budget(config):\n        raise RuntimeError(f"Snapshot exceeds Output budget; raw snapshot remains in this runtime at {raw}")\n    suffix = ".snapshot.gz" if source.suffix == ".gz" else ".snapshot"\n    target = output / (prefix + suffix)\n    if target.exists():\n        raise FileExistsError("Export already exists; choose a fresh output directory")\n    partial = target.with_suffix(target.suffix + ".partial")\n    shutil.copyfile(source, partial)\n    manifest.update(snapshot_file=target.name, snapshot_bytes=partial.stat().st_size,\n                    snapshot_sha256=checksum(partial))\n    if manifest["snapshot_sha256"] != checksum(source):\n        raise ValueError("Export checksum mismatch")\n    partial.replace(target)\n    write_json(output / "ingestion_checkpoint.json", state)\n    write_json(output / "snapshot_manifest.json", manifest)\n    write_json(output / "verification_report.json", manifest["restore_verification"])\n    print(json.dumps({"phase": "snapshot_exported", "complete": complete,\n                      "points": points, "file": target.name, "bytes": manifest["snapshot_bytes"]}), flush=True)\n    return manifest\n\n\ndef restore_for_resume(config, identity):\n    folder = Path(config["resume_from"])\n    manifest = json.loads((folder / "snapshot_manifest.json").read_text())\n    if manifest["signature"] != identity or manifest["qdrant_version"] != QDRANT_VERSION:\n        raise ValueError("Resume snapshot belongs to a different input/configuration")\n    snapshot = folder / manifest["snapshot_file"]\n    if checksum(snapshot) != manifest["snapshot_sha256"]:\n        raise ValueError("Resume snapshot checksum mismatch")\n    raw = Path(config["state_dir"]) / "snapshots" / "resume.snapshot"\n    opener = gzip.open if snapshot.suffix == ".gz" else open\n    with opener(snapshot, "rb") as source, raw.open("wb") as destination:\n        shutil.copyfileobj(source, destination, 8 * 1024**2)\n    if checksum(raw) != manifest["raw_snapshot_sha256"]:\n        raise ValueError("Unpacked resume checksum mismatch")\n    api(config, "PUT", f"/collections/{config[\'collection\']}/snapshots/recover?wait=true",\n        {"location": raw.resolve().as_uri(), "priority": "snapshot"}, timeout=3600)\n    if count_points(config) != manifest["points"]:\n        raise ValueError("Restored snapshot exact count differs")\n    samples = json.loads((folder / "verification_samples.json").read_text())\n    verify_collection(config["qdrant_url"], manifest, samples)\n    write_json(Path(config["output"]) / "verification_samples.json", samples)\n    state = manifest["checkpoint"]\n    write_json(checkpoint_path(config), state)\n    return state\n\n\ndef smoke_test(config, catalog):\n    smoke = {**config, "collection": config["collection"] + "_preflight"}\n    first = catalog["files"][0]\n    points, _ = next(iter_points(first["path"], first["receipt"], config["dimension"], 3))\n    create_collection(smoke)\n    try:\n        with requests.Session() as session:\n            upsert(smoke, points, session)\n        identity = signature(config, catalog)\n        manifest = snapshot_manifest(smoke, identity, {}, len(points), False)\n        samples = [{"id": p["id"], "payload": p["payload"], "vector": p["vector"][VECTOR_NAME]} for p in points]\n        raw, _ = snapshot_raw(smoke)\n        verify_snapshot_restore(smoke, raw, manifest, samples)\n        api(smoke, "DELETE", f"/collections/{smoke[\'collection\']}/snapshots/{raw.name}")\n        print("PREFLIGHT PASSED: real FP16 upsert, payload lookup, snapshot and restore", flush=True)\n    finally:\n        api(smoke, "DELETE", f"/collections/{smoke[\'collection\']}?timeout=120")\n\n\ndef run_pipeline(config):\n    started = time.monotonic()\n    output, state_dir = Path(config["output"]), Path(config["state_dir"])\n    for path in (output, state_dir, Path(config["output_budget_root"])):\n        path.mkdir(parents=True, exist_ok=True)\n    if (output / "snapshot_manifest.json").exists():\n        raise FileExistsError("Export exists; use a fresh Output directory")\n    process = log = None\n    collection_ready = False\n    identity = state = None\n    try:\n        write_json(output / "pipeline_status.json", {"success": False, "stage": "validating_inputs"})\n        catalog = load_catalog(config)\n        identity = signature(config, catalog)\n        saved = state_dir / "run_signature.json"\n        if saved.exists() and json.loads(saved.read_text()) != identity:\n            raise ValueError("State belongs to another input/configuration")\n        write_json(saved, identity)\n        write_json(output / "input_catalog.json", catalog)\n        write_json(output / "encoder_profile.json", catalog["encoder"])\n        check_resources(config, catalog)\n        process, log = start_server(config)\n        existing = api(config, "GET", "/collections")["collections"]\n        state = {"identity": identity, "rows": 0, "file_index": 0, "file_rows": 0, "phase": "new"}\n        if config.get("resume_from"):\n            if existing:\n                raise ValueError("Resume requires new empty state, not an existing database")\n            state = restore_for_resume(config, identity)\n        elif checkpoint_path(config).exists():\n            state = json.loads(checkpoint_path(config).read_text())\n            if state["identity"] != identity:\n                raise ValueError("Checkpoint identity mismatch")\n        elif existing:\n            raise ValueError("Existing database without matching checkpoint; restore the matching snapshot")\n        if not existing and not config.get("resume_from"):\n            if state["rows"]:\n                raise ValueError("Checkpoint exists but database is missing; attach matching recovery snapshot")\n            smoke_test(config, catalog)\n        create_collection(config)\n        collection_ready = True\n        count = count_points(config)\n        if count < state["rows"] or count > state["rows"] + config["batch_size"]:\n            raise ValueError("Database count inconsistent with committed checkpoint")\n        write_json(checkpoint_path(config), state)\n        write_json(output / "pipeline_status.json", {"success": False, "stage": "ingesting"})\n        ingest(config, catalog, state, started)\n        # Keep this checkpoint snapshot in runtime scratch; export recovery on controlled failure.\n        # This avoids keeping two full snapshots in the limited Kaggle Output directory.\n        pre_index, _ = snapshot_raw(config)\n        print(f"Pre-index runtime snapshot: {pre_index}; not yet durable across Kaggle sessions", flush=True)\n        state["phase"] = "indexing"\n        write_json(checkpoint_path(config), state)\n        write_json(output / "pipeline_status.json", {"success": False, "stage": "indexing"})\n        wait_for_index(config)\n        samples = json.loads((output / "verification_samples.json").read_text())\n        manifest = snapshot_manifest(config, identity, state, catalog["rows"], True)\n        verify_collection(config["qdrant_url"], manifest, samples)\n        write_json(output / "dense_query_example.json", {"dense_vector": samples[0]["vector"],\n                   "chunk_id": samples[0]["payload"]["chunk_id"],\n                   "note": "Stored-vector smoke test. Encode new queries with the pinned Qwen3 encoder."})\n        state["phase"] = "indexed"\n        write_json(checkpoint_path(config), state)\n        export_snapshot(config, identity, state, complete=True)\n        write_json(output / "pipeline_status.json", {"success": True, "stage": "snapshot_exported"})\n    except (Exception, KeyboardInterrupt) as error:\n        status = {"success": False, "stage": state["phase"] if state else "preflight", "error": str(error)}\n        if collection_ready and state is not None:\n            try:\n                if count_points(config) and (output / "verification_samples.json").exists():\n                    export_snapshot(config, identity, state, complete=False)\n                    status["recovery_exported"] = True\n            except Exception as recovery_error:\n                status["recovery_error"] = str(recovery_error)\n                print("Recovery export failed:", str(recovery_error), flush=True)\n        write_json(output / "pipeline_status.json", status)\n        print("PIPELINE FAILED:", json.dumps(status), flush=True)\n        raise\n    finally:\n        if process is not None:\n            stop_server(process, log)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, required=True)\n    args = parser.parse_args()\n    run_pipeline(json.loads(args.config.read_text()))\n', 'check_vibiomir_snapshot.py': '"""Restore and revalidate the existing ViBioMIR recovery snapshot on Kaggle CPU."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport statistics\nimport time\n\nimport psutil\nimport requests\n\nif __package__:\n    from . import vibiomir_qdrant_runtime as runtime\n    from .restore_vibiomir_qdrant import verify_collection\nelse:\n    import vibiomir_qdrant_runtime as runtime\n    from restore_vibiomir_qdrant import verify_collection\n\n\ndef find_source(config):\n    candidates = []\n    for path in Path(config["input_root"]).rglob("snapshot_manifest.json"):\n        manifest = json.loads(path.read_text())\n        if manifest.get("schema_version") != "vibiomir-qdrant-dense-v1":\n            continue\n        if manifest["signature"]["source_sha256"] != config["source_sha256"]:\n            continue\n        candidates.append((path, manifest))\n    if len(candidates) != 1:\n        raise ValueError(f"Attach exactly one original dense snapshot output; found {len(candidates)}")\n    path, manifest = candidates[0]\n    signature = manifest["signature"]\n    if (manifest["points"] != config["expected_chunks"]\n            or manifest["checkpoint"]["rows"] != config["expected_chunks"]\n            or signature["profile_id"] != config["profile_id"]\n            or signature["dimension"] != config["dimension"]\n            or signature["hnsw_m"] != config["hnsw_m"]\n            or signature["ef_construction"] != config["ef_construction"]\n            or manifest["qdrant_version"] != runtime.QDRANT_VERSION\n            or manifest["vector_schema"] != {"dense": "qwen3_dense", "dense_datatype": "float16"}\n            or Path(manifest["snapshot_file"]).name != manifest["snapshot_file"]):\n        raise ValueError("Snapshot does not match the full-corpus FP16 HNSW contract")\n    snapshot = path.parent / manifest["snapshot_file"]\n    if snapshot.suffix != ".snapshot" or snapshot.stat().st_size != manifest["raw_snapshot_bytes"]:\n        raise ValueError("This bounded recheck expects the original raw .snapshot output")\n    if manifest["snapshot_sha256"] != manifest["raw_snapshot_sha256"]:\n        raise ValueError("Raw and exported snapshot hashes differ")\n    samples = json.loads((path.parent / "verification_samples.json").read_text())\n    if not samples:\n        raise ValueError("Source verification samples are missing")\n    return path, manifest, snapshot, samples\n\n\ndef stage_snapshot(source, destination, expected_hash):\n    digest = hashlib.sha256()\n    total = 0\n    with Path(source).open("rb") as reader, Path(destination).open("wb") as writer:\n        for block in iter(lambda: reader.read(8 * 1024**2), b""):\n            digest.update(block)\n            writer.write(block)\n            total += len(block)\n            if total % (1024**3) == 0:\n                print(json.dumps({"stage": "staging_snapshot", "bytes": total}), flush=True)\n    if digest.hexdigest() != expected_hash or runtime.checksum(destination) != expected_hash:\n        raise ValueError("Source or staged snapshot SHA-256 mismatch")\n    print(json.dumps({"stage": "snapshot_sha256_verified", "bytes": total,\n                      "sha256": expected_hash}), flush=True)\n\n\ndef measured_search(config, sample, *, server_timeout=None, filtered=False, label="search"):\n    query = {"query": sample["vector"], "using": "qwen3_dense", "limit": 10,\n             "with_payload": True, "params": {"hnsw_ef": 128}}\n    if filtered:\n        query["filter"] = {"must": [{"key": "content_origin",\n                                     "match": {"value": sample["payload"]["content_origin"]}}]}\n    params = {"timeout": server_timeout} if server_timeout is not None else None\n    started = time.monotonic()\n    result = {"label": label, "server_timeout": server_timeout, "filtered": filtered,\n              "sample_chunk_id": sample["payload"]["chunk_id"]}\n    try:\n        response = requests.post(f"{config[\'qdrant_url\']}/collections/{config[\'collection\']}/points/query",\n                                 params=params, json=query, timeout=(server_timeout or 60) + 30)\n        result["http_status"] = response.status_code\n        if response.ok:\n            hits = response.json()["result"]["points"]\n            result.update(hits=len(hits), top_score=hits[0]["score"] if hits else None,\n                          returned_ids=[str(hit["id"]) for hit in hits],\n                          payload_present=all("content" in hit.get("payload", {})\n                                              and "_embedding" in hit["payload"] for hit in hits))\n            result["passed"] = bool(hits and hits[0]["score"] >= 0.99 and result["payload_present"])\n            if filtered:\n                result["passed"] = result["passed"] and all(\n                    hit["payload"]["content_origin"] == sample["payload"]["content_origin"] for hit in hits)\n        else:\n            result.update(passed=False, error=response.text[:3000])\n    except requests.RequestException as error:\n        result.update(passed=False, error=str(error))\n    result["elapsed_seconds"] = round(time.monotonic() - started, 4)\n    print(json.dumps({"stage": "query_check", **result}), flush=True)\n    return result\n\n\ndef run(config):\n    output, state = Path(config["output"]), Path(config["state_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n    state.mkdir(parents=True, exist_ok=True)\n    report = {"passed": False, "stage": "preflight", "queries": [], "reingested": False,\n              "reembedded": False, "source_kernel": config["source_kernel"]}\n    process = log = None\n    try:\n        manifest_path, manifest, source, samples = find_source(config)\n        if any(state.iterdir()):\n            raise ValueError("Use a fresh empty state directory for the recheck")\n        mounts = [m for m in psutil.disk_partitions(all=True)\n                  if state.resolve().is_relative_to(Path(m.mountpoint))]\n        kind = max(mounts, key=lambda m: len(m.mountpoint)).fstype if mounts else "unknown"\n        if kind in {"tmpfs", "ramfs"}:\n            raise RuntimeError(f"Restore state must be disk-backed, not {kind}")\n        required = 3 * source.stat().st_size + 4 * 1024**3\n        free = shutil.disk_usage(state).free\n        if free < required:\n            raise RuntimeError(f"Insufficient scratch disk: need {required}, free {free}")\n        print(json.dumps({"stage": "preflight", "snapshot_bytes": source.stat().st_size,\n                          "state_free_bytes": free, "required_state_bytes": required,\n                          "memory_available_bytes": psutil.virtual_memory().available,\n                          "filesystem": kind, "source_complete_flag": manifest["complete"]}), flush=True)\n        runtime.write_json(output / "source_snapshot_manifest.json", manifest)\n        runtime.write_json(output / "verification_samples.json", samples)\n        runtime.write_json(output / "run_config.json", config)\n        process, log = runtime.start_server(config)\n        if runtime.api(config, "GET", "/collections")["collections"]:\n            raise ValueError("Recheck server is not empty")\n        raw = state / "snapshots" / "source.snapshot"\n        report["stage"] = "staging_snapshot"\n        runtime.write_json(output / "recheck_report.json", report)\n        stage_snapshot(source, raw, manifest["raw_snapshot_sha256"])\n        report["stage"] = "restoring"\n        runtime.write_json(output / "recheck_report.json", report)\n        print(json.dumps({"stage": "restoring", "collection": config["collection"]}), flush=True)\n        restore_started = time.monotonic()\n        runtime.api(config, "PUT", f"/collections/{config[\'collection\']}/snapshots/recover?wait=true",\n                    {"location": raw.resolve().as_uri(), "priority": "snapshot"}, timeout=7200)\n        report["restore_seconds"] = round(time.monotonic() - restore_started, 3)\n        info = runtime.api(config, "GET", f"/collections/{config[\'collection\']}")\n        runtime.write_json(output / "restored_collection_info.json", info)\n        if (info["status"] != "green" or info["optimizer_status"] != "ok"\n                or info.get("indexed_vectors_count") != config["expected_chunks"]):\n            raise ValueError("Restored collection is not green and fully HNSW-indexed")\n        report["stage"] = "checking_search"\n        runtime.write_json(output / "recheck_report.json", report)\n        print(json.dumps({"stage": "restored", "restore_seconds": report["restore_seconds"],\n                          "status": info["status"], "indexed_vectors_count": info["indexed_vectors_count"]}), flush=True)\n        # Test the original server default separately; an error here must not hide long-timeout results.\n        report["queries"].append(measured_search(config, samples[0], label="first_query_default_60s"))\n        runtime.write_json(output / "recheck_report.json", report)\n        # Complete=True here describes the verification expectation, not a modification of the input manifest.\n        expected = {**manifest, "complete": True}\n        report["verification"] = verify_collection(config["qdrant_url"], expected, samples,\n            collection=config["collection"], search_timeout=config["search_timeout_seconds"])\n        print(json.dumps({"stage": "schema_payload_vector_lookup_verified", **report["verification"]}), flush=True)\n        stride = max(1, len(samples) // 9)\n        for index, sample in enumerate(samples[::stride][:9]):\n            report["queries"].append(measured_search(config, sample,\n                server_timeout=config["search_timeout_seconds"], label=f"explicit_timeout_{index}"))\n            runtime.write_json(output / "recheck_report.json", report)\n        for index, sample in enumerate(samples[::max(1, len(samples)//3)][:3]):\n            report["queries"].append(measured_search(config, sample,\n                server_timeout=config["search_timeout_seconds"], filtered=True, label=f"origin_filter_{index}"))\n            runtime.write_json(output / "recheck_report.json", report)\n        required_checks = report["queries"][1:]\n        if not all(check["passed"] for check in required_checks):\n            raise ValueError("One or more explicit-timeout dense/filter queries failed; see recheck_report.json")\n        seconds = [check["elapsed_seconds"] for check in required_checks]\n        report.update(passed=True, stage="verified", points=config["expected_chunks"],\n                      source_snapshot_sha256=manifest["raw_snapshot_sha256"],\n                      source_snapshot_file=manifest["snapshot_file"],\n                      source_manifest_complete=manifest["complete"],\n                      query_median_seconds=round(statistics.median(seconds), 4),\n                      query_max_seconds=max(seconds), default_query_passed=report["queries"][0]["passed"],\n                      note="Functional validation of stored-vector queries, not semantic retrieval quality evaluation.")\n        certificate = {"validated": True, "source_kernel": config["source_kernel"],\n                       "source_snapshot_file": manifest["snapshot_file"],\n                       "source_snapshot_sha256": manifest["raw_snapshot_sha256"],\n                       "points": config["expected_chunks"], "qdrant_version": runtime.QDRANT_VERSION,\n                       "signature": manifest["signature"], "restore_verification": report["verification"],\n                       "search_timeout_seconds": config["search_timeout_seconds"],\n                       "original_complete_flag": manifest["complete"],\n                       "note": "This certificate validates the referenced immutable recovery snapshot; original manifest remains unchanged."}\n        runtime.write_json(output / "snapshot_validation.json", certificate)\n        runtime.write_json(output / "recheck_report.json", report)\n        print(json.dumps({"stage": "RECHECK_PASSED", "points": report["points"],\n                          "default_query_passed": report["default_query_passed"],\n                          "query_median_seconds": report["query_median_seconds"],\n                          "query_max_seconds": report["query_max_seconds"]}), flush=True)\n    except Exception as error:\n        report.update(passed=False, error=str(error))\n        runtime.write_json(output / "recheck_report.json", report)\n        print(json.dumps({"stage": "RECHECK_FAILED", "error": str(error)}), flush=True)\n        raise\n    finally:\n        if process is not None:\n            runtime.stop_server(process, log)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, required=True)\n    args = parser.parse_args()\n    run(json.loads(args.config.read_text()))\n'}
runner = Path('/tmp/vibiomir_snapshot_check_runner')
runner.mkdir(parents=True, exist_ok=True)
output = Path(CONFIG['output'])
output.mkdir(parents=True, exist_ok=True)
for name, source in MODULES.items():
    compile(source, name, 'exec')
    (runner / name).write_text(source)
config_path = runner / 'config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))


## Run the check

Scratch needs disk for the staged 18.82 GB snapshot and restored collection. Output contains small reports and a hash-bound validation certificate. The original recovery snapshot and original complete=false manifest remain unchanged.


In [ ]:
command = [sys.executable, '-u', str(runner / 'check_vibiomir_snapshot.py'), '--config', str(config_path)]
with (output / 'recheck.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
report_path = output / 'recheck_report.json'
report = json.loads(report_path.read_text()) if report_path.exists() else {'passed': False, 'stage': 'startup'}
print('FINAL REPORT:', json.dumps({k: v for k, v in report.items() if k != 'queries'}, indent=2))
if returncode or not report.get('passed'):
    raise RuntimeError('Snapshot recheck failed; inspect recheck_report.json and qdrant.log')
